In [2]:
import numpy as np
import scipy.sparse
import matplotlib.pyplot as plt
from pathlib import Path
import multiprocessing as mp
import roicat
%load_ext autoreload
%autoreload 2
import roicat.util
import seaborn as sns
import tempfile
import os
import analysis_function as af
import matplotlib.image as mpimg
from IPython.display import clear_output
from scipy.ndimage import gaussian_filter1d

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Suite2p data structure
# SOM_R --- (AB1/AB3/AB5/C/D...) --- (behavior_data/raw_data/red/suite2p) --- (plane0/plane1/plane2/plane3) --- (iscell.npy/F.npy/Fneu.npy...)

In [94]:
dir_input = r"Y:\public\projects\SaEl_20220201_VIP\2pdata\RSCsilencing\Halo\SOM\SOM_LL" # input folder path
dir_save = r"D:\suite2p alignment\SOM_LL\results"
sessions_to_align = ['AB1','AB3','F']  # list that includes sessions to align
um_per_pixel = 0.4   # Micrometer per pixel of image field of view
n_sessions_minimum = 3  # rois that are identified in less than this number of sessions will be removed

# af.generate_aligned_FOV_images(dir_input, um_per_pixel, sessions_to_align, dir_save)

In [ ]:
# SOM1 - AB1, AB3 (or AB5, whichever yields more cells), D sessions
# SOM_LL - AB1, AB3 (or AB5, whichever yields more cells), F sessions

In [88]:
aligned_data_all = {}
planes = []
for subfolder in os.listdir(dir_save):
    subfolder_path = os.path.join(dir_save, subfolder)
    planes.append(Path(subfolder_path).name)

for plane in planes:
    print(f'Align data for {plane}:')
    aligned_data_all[plane] = []
    subfolder_path = os.path.join(dir_save, plane)
    print('')
    print(f'First, check if the images are aligned')
    print('')
    aligned_img_path = os.path.join(subfolder_path, f'aligned_img_{plane}.npy')
    alignment_scores_path = os.path.join(subfolder_path, f'alignment_scores_{plane}.png')
    print('The left panel shows the alignment scores (z-scores) computed for each pair of images. It represents how many radius_out std the radius_in phase-correlation peak(max) exceeds the 95th-percentile background(radius_out torus)')
    print('The right panel shows the same thing masked by the Z_threshold (10 std), alignment scores above the threshold is yellow and below threshold is purple')
    plt.imshow(mpimg.imread(alignment_scores_path))
    plt.tight_layout() 
    plt.axis('off')
    plt.show()
    print('')
    # print('An example similarity map that shows radius_in and radius_out, x and y are the offset of the two images')
    # af.toy_similarity_map()
    # print(f'Also view the toggle image')
    roicat.visualization.display_toggle_image_stack(np.load(aligned_img_path, allow_pickle=True), image_size=0.8)

    # if aligned, continue processing, first load the alignment results
    Rich_File_Path = os.path.join(subfolder_path, 'result.tracking.results_all.richfile')
    results = roicat.util.RichFile_ROICaT(path=Rich_File_Path).load()

    # then load the fluorescence data and get z_scored DF/F
    dF_F = []
    raw_fluorescence_data = []
    for session in sessions_to_align:
        F = np.load(os.path.join(dir_input, session, 'suite2p',plane,'F.npy'))
        Fneu = np.load(os.path.join(dir_input, session, 'suite2p',plane,'Fneu.npy'))
        TC = (F) - (0.7 * (Fneu))  # 0.7 is the neuropil factor
        TC_smoothed = gaussian_filter1d(TC, sigma=0.7, axis=1) 
        baseline = np.percentile(TC_smoothed,50,axis=1)
        dF = TC_smoothed - baseline[:, None]
        TC_dff = dF.T / baseline
        TC_dff_Zscore = (TC_dff - np.nanmedian(TC_dff, axis=0)) / np.nanstd(TC_dff, axis=0)
        dF_F.append(TC_dff_Zscore.T)
        raw_fluorescence_data.append(F)
    fluorescence_data = dF_F
    
    # apply the iscell mask
    iscell = []
    for session in sessions_to_align:
        iscell_path = os.path.join(dir_input, session, 'suite2p',plane,'iscell.npy')
        list = []
        for d in np.load(iscell_path):
            if d[0] ==  1: 
                list.append(True)
            else:
                list.append(False) 
        iscell.append(list) # add all iscell data to a list
    roi_labels = results['clusters']['labels_bySession']
    labels_iscell = roicat.util.mask_UCIDs_with_iscell(    ## Get the iscell masked labels
        ucids=roi_labels,
        iscell=iscell,
    )
    labels_iscell = roicat.util.squeeze_UCID_labels(ucids=labels_iscell, return_array=True)  ## [(n_rois,)] * n_sessions
    data_aligned_masked_iscell = roicat.util.match_arrays_with_ucids(     ## Align the data with the iscell masks
        arrays= fluorescence_data,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois, n_timepoints))
        ucids=labels_iscell,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois,))  OR   concatenated numpy array (shape (n_rois_total,))
    )

    # Quality Control
    cluster_sihouette_threshold = 0.2  # Recommended thresholds
    sample_sihouette_threshold = 0.1    
    labels_keep_1 = []  # labels of rois that pass the cluster_sihouette threshold
    labels_keep_2 = []  # labels of rois that pass the sample_sihouette_threshold
    for idx in range(len(results['clusters']['quality_metrics']['cluster_silhouette'])):
        if results['clusters']['quality_metrics']['cluster_silhouette'][idx] > cluster_sihouette_threshold:
            labels_keep_1.append(idx-1)  # because the roi cluster index starts from -1, this is correct
    for idx in range(len(results['clusters']['quality_metrics']['sample_silhouette'])):
        if results['clusters']['quality_metrics']['sample_silhouette'][idx] > sample_sihouette_threshold:
            labels_keep_2.append(results['clusters']['labels'][idx])
    labels_keep_1 = np.unique(labels_keep_1)
    labels_keep_2 = np.unique(labels_keep_2)
    labels_keep = np.intersect1d(np.array(labels_keep_1), np.array(labels_keep_2)) # get the cluster labels that pass both threshold
    labels_masked_keep = roicat.util.mask_UCIDs_by_label(  # 'labels' ---> 'labels_bySession'
        ucids=labels_iscell, 
        labels=labels_keep,
    )
    labels_masked_keep = roicat.util.squeeze_UCID_labels(ucids=labels_masked_keep, return_array=True)  ## [(n_rois,)] * n_sessions
    data_aligned_masked_keep = roicat.util.match_arrays_with_ucids(  
        arrays=fluorescence_data,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois, n_timepoints))
        ucids=labels_masked_keep,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois,))  OR   concatenated numpy array (shape (n_rois_total,))
    )


    # Mask by min_sessions   
    labels_minSesh = roicat.util.discard_UCIDs_with_fewer_matches(   ## Keep labels / UCIDs that are assigned in at least [n_sessions_minimum] sessions
        ucids=labels_masked_keep,
        n_sesh_thresh=n_sessions_minimum,
    )
    labels_minSesh = roicat.util.squeeze_UCID_labels(ucids=labels_minSesh, return_array=True)  ## [(n_rois,)] * n_sessions
    ## Align the data with the masked labels
    data_aligned_minSesh = roicat.util.match_arrays_with_ucids(
        arrays=fluorescence_data,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois, n_timepoints))
        ucids=labels_minSesh,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois,))  OR   concatenated numpy array (shape (n_rois_total,))
    )

   



    ## Align the data with the masked labels
    data_aligned_minSesh = roicat.util.match_arrays_with_ucids(
        arrays=raw_fluorescence_data,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois, n_timepoints))
        ucids=labels_minSesh,  ## expects list (length n_sessions) of numpy arrays (shape (n_rois,))  OR   concatenated numpy array (shape (n_rois_total,))
    )
    

    print('')
    print('')
    print('Check the results:')
    print("Original data shapes [(n_rois, n_timepoints)] * n_sessions:")
    print([d.shape for d in fluorescence_data])
    print("Iscell masked aligned data shapes:")
    print([d.shape for d in data_aligned_masked_iscell])
    print("Quality metrics masked aligned data shapes:")
    print([d.shape for d in data_aligned_masked_keep])
    print("Labels_minSesh masked aligned data shapes:")
    print([d.shape for d in data_aligned_minSesh])
    print('')
    percentage_remain = (data_aligned_minSesh[0].shape[0]/data_aligned_masked_iscell[0].shape[0])*100
    print(f'Summary1: {int(percentage_remain)}% of rois remain after masked by quality metrics and min_sessions')

    # print('Cluster_silhouette: A measure of how similar an ROI is to its own cluster compared to other clusters, which can be indicative of the appropriateness of the cluster assignment.')
    # print('Sample_sihouette: A measure of how well each ROI is clustered with its label, providing a perspective on the overall clustering quality.')
    # fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10,5)) 
    # axs[0].hist(results['clusters']['quality_metrics']['cluster_silhouette'], 50); 
    # axs[0].set_xlabel('cluster_silhouette');
    # axs[0].set_ylabel('cluster counts');
    # axs[0].axvline(0.2,color='black',linestyle='--')
    # axs[1].hist(results['clusters']['quality_metrics']['sample_silhouette'], 50);
    # axs[1].set_xlabel('sample_silhouette score');
    # axs[1].set_ylabel('roi sample counts');  
    # axs[1].axvline(0.2,color='black',linestyle='--')
    # axs[1].hist(confidence, 50);
    # axs[1].set_xlabel('confidence');
    # axs[1].set_ylabel('cluster counts');
    # axs[1].axvline(0.6,color='black',linestyle='--')
    # plt.show() 
    
    # Calculate the confidence scores
    confidence = (((np.array(results['clusters']['quality_metrics']['cluster_silhouette']) + 1) / 2) * np.array(results['clusters']['quality_metrics']['cluster_intra_means']))
    top_conf = np.sort(confidence)[-data_aligned_minSesh[0].shape[0]:]
    median_conf = np.median(top_conf)
    perc = np.mean(top_conf > 0.6) * 100
    print(f'Summary2: The median Confidence score for the remaining cells is {median_conf:.2f}, Percentage of cells above 0.6 confidence score: {perc:.1f}%')
    print('')
    print(f'Note: High confidence (e.g. >0.6) means that the cluster is well-separated and internally coherent. Low confidence means either the cluster overlaps with others, or the cluster is noisy inside (low intra similarity), or both')
    print('')   



    # print('Show the aligned fluorescence data (z_scored df/f), top one is before alignment and bottom one is after alignment')
    # fig, axs = plt.subplots(2, len(sessions_to_align), figsize=(10, 5))
    # for i in range(len(sessions_to_align)):
    #     im0 = axs[0, i].imshow(fluorescence_data[i], aspect="auto", cmap="bwr",
    #                         vmin=np.percentile(fluorescence_data[i], 1),
    #                         vmax=np.percentile(fluorescence_data[i], 99),
    #                         interpolation="none")
    #     im1 = axs[1, i].imshow(data_aligned_minSesh[i], aspect="auto", cmap="bwr",
    #                         vmin=np.percentile(data_aligned_minSesh[i], 1),
    #                         vmax=np.percentile(data_aligned_minSesh[i], 99),
    #                         interpolation="none")
    #     axs[0, i].set_title(f"Session {i+1}")
    #     axs[1, i].set_title(f"Session {i+1} (aligned)")
    #     axs[0, i].set_xlabel("Timepoints")
    #     axs[1, i].set_xlabel("Timepoints")
    #     axs[0, i].set_ylabel("ROIs")
    #     axs[1, i].set_ylabel("ROIs")
    #     if i == len(sessions_to_align) - 1:
    #         fig.colorbar(im0, ax=axs[0, i], label="dF/F")
    #         fig.colorbar(im1, ax=axs[1, i], label="dF/F")
    # plt.tight_layout()
    # plt.show()



    print('Show the aligned fluorescence raw data, top one is before alignment and bottom one is after alignment')
    fig, axs = plt.subplots(2, len(sessions_to_align), figsize=(15, 5))
    for i in range(len(sessions_to_align)):
        axs[0, i].imshow(raw_fluorescence_data[i], aspect="auto", cmap="rainbow", interpolation="none")
        axs[1, i].imshow(data_aligned_minSesh[i], aspect="auto", cmap="rainbow", interpolation="none")
        axs[0, i].set_title(f"Session {i+1}")
        axs[1, i].set_title(f"Session {i+1} (aligned)")
        axs[0, i].set_xlabel("Timepoints")
        axs[1, i].set_xlabel("Timepoints")
        axs[0, i].set_ylabel("ROIs")
        axs[1, i].set_ylabel("ROIs")
        ## Colorbar
        if i == len(sessions_to_align) - 1:
            fig.colorbar(axs[0, i].imshow(raw_fluorescence_data[i], aspect="auto", cmap="rainbow", interpolation="none"), ax=axs[0, i], label="Fluorescence (a.u.)")
            fig.colorbar(axs[1, i].imshow(data_aligned_minSesh[i], aspect="auto", cmap="rainbow", interpolation="none"), ax=axs[1, i], label="Fluorescence (a.u.)")
    plt.tight_layout()
    plt.show()


    print('View the toggle image with aligned ROIs colored')
    labels = []
    for session in labels_minSesh:
        for label in session:
            labels.append(label)
    labels = np.array(labels)
    FOVs_colored = roicat.visualization.compute_colored_FOV(
        labels=labels,
        spatialFootprints=results['ROIs']['ROIs_aligned'], 
        FOV_height=results['ROIs']['frame_height'], 
        FOV_width=results['ROIs']['frame_width'], 
    )
    combined_images = [
        af.overlay_images(roicat_img, fov_img, alpha=0.6)
        for roicat_img, fov_img in zip(np.load(aligned_img_path, allow_pickle=True), FOVs_colored)]
    roicat.visualization.display_toggle_image_stack(combined_images)

    print('check if ROIs in the same cluster have similar shapes')
    num =  int(input('how many rois do you want to check? e.g. 20'))
    ucids = []
    for session in labels_minSesh:
        for label in session:
            ucids.append(label)
    ucids = np.array(ucids)
    ucids_unique = np.unique(ucids[ucids>=0])
    ROI_ims_sparse = scipy.sparse.vstack(results['ROIs']['ROIs_aligned'])
    ROI_ims_sparse = ROI_ims_sparse.multiply( ROI_ims_sparse.max(1).power(-1) ).tocsr()
    ucid_sfCat = []  # image for all ucids stored in this list
    for ucid in ucids_unique:
        idx = np.where(ucids == ucid)[0]
        ucid_sfCat.append(np.concatenate(roicat.visualization.crop_cluster_ims(ROI_ims_sparse[idx].toarray().reshape(len(idx), results['ROIs']['frame_height'], results['ROIs']['frame_width'])), axis=1) )
    for ii in range(min(len(ucid_sfCat), num)): # the number here is the number of ROIs to display
        plt.figure(figsize=(40,1))
        plt.imshow(ucid_sfCat[ii], cmap='gray')
        plt.axis('off')
        plt.show()

    # The end of processing for one plane
    answer = input(f"Do you want to keep results for {plane}? (Y/N): ").strip().lower()
    if answer == "y":
        aligned_data_all[plane] = data_aligned_minSesh
        print(f"✅ Results for {plane} added.")
    elif answer == "n":
        print(f"❌ Results for {plane} skipped.")
    clear_output(wait=True)  # clear the output from last cycle
    continue

print('The final result can be found in: aligned_data_all, the keys are the planes')

Align data for plane3:

First, check if the images are aligned

The left panel shows the alignment scores (z-scores) computed for each pair of images. It represents how many radius_out std the radius_in phase-correlation peak(max) exceeds the 95th-percentile background(radius_out torus)
The right panel shows the same thing masked by the Z_threshold (10 std), alignment scores above the threshold is yellow and below threshold is purple


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\suite2p alignment\\SOM_LL\\results\\plane3\\alignment_scores_plane3.png'

In [89]:
aligned_data_all.keys()

dict_keys(['plane0', 'plane1', 'plane2', 'plane3'])

In [95]:
# Save F_aligned to corresponding folders

for plane in aligned_data_all:
    for i,session in enumerate(sessions_to_align):
        path = os.path.join(dir_input,session,'suite2p',plane)
        save_path = os.path.join(path, 'F_aligned')
        if aligned_data_all[plane] == []:
            np.save(save_path, [])
        else:
            np.save(save_path, aligned_data_all[plane][i])

FileNotFoundError: [Errno 2] No such file or directory: 'Y:\\public\\projects\\SaEl_20220201_VIP\\2pdata\\RSCsilencing\\Halo\\SOM\\SOM_LL\\F\\suite2p\\plane3\\F_aligned.npy'